In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

from models.LogisticRegression import LogisticRegressionPytorch

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# instantiate dataloader to grab data from ELK
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='lexical'
)

In [3]:
X = dataloader.df.drop(['url', 'type'], axis=1, inplace=False) # url not needed since not discrete/continuous feature, label removed
y = dataloader.df['type'] # store label

X_train, X_test, y_train, y_test = dataloader.train_test_split(X, y, train_split=0.8, random_state=None) # Pareto principle split for train/test sets

In [4]:
X_train

,lexical_len_url,lexical_len_netloc,lexical_count_digits_netloc,lexical_count_letters_netloc,lexical_ratio_digits_netloc_url,lexical_ratio_letters_netloc_url,lexical_len_path,lexical_count_digits_path,lexical_count_letters_path,lexical_ratio_digits_path_url,...,lexical_count_dots_url,lexical_count_percent_url,lexical_count_hash_url,lexical_count_ats_url,lexical_count_embed_url,lexical_use_https,lexical_no_of_directories,lexical_contains_ip_address,lexical_character_continuity_rate_url,lexical_shannon_entropy_url
4630,15,0,0,0,0.000000,0.000000,15,1,13,0.066667,...,1,0,0,0,0,0,0,0,0.000000,3.906891
9915,17,0,0,0,0.000000,0.000000,17,0,16,0.000000,...,1,0,0,0,0,0,0,0,0.000000,3.616875
7409,15,0,0,0,0.000000,0.000000,15,0,14,0.000000,...,1,0,0,0,0,0,0,0,0.000000,3.640224
2063,9,0,0,0,0.000000,0.000000,9,1,7,0.111111,...,1,0,0,0,0,0,0,0,0.000000,2.947703
8573,11,0,0,0,0.000000,0.000000,11,0,10,0.000000,...,1,0,0,0,0,0,0,0,0.000000,2.845351
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5483,39,30,0,28,0.000000,0.717949,1,0,0,0.000000,...,2,0,0,0,1,1,1,0,0.076923,4.304257
6892,19,0,0,0,0.000000,0.000000,19,0,18,0.000000,...,1,0,0,0,0,0,0,0,0.000000,3.681881
4832,14,0,0,0,0.000000,0.000000,14,0,12,0.000000,...,2,0,0,0,0,0,0,0,0.000000,3.378783
8076,84,76,11,61,0.130952,0.726190,1,0,0,0.000000,...,3,0,0,0,1,0,1,0,0.047619,4.853527


In [5]:
list(X_train.columns)

['lexical_len_url',
 'lexical_len_netloc',
 'lexical_count_digits_netloc',
 'lexical_count_letters_netloc',
 'lexical_ratio_digits_netloc_url',
 'lexical_ratio_letters_netloc_url',
 'lexical_len_path',
 'lexical_count_digits_path',
 'lexical_count_letters_path',
 'lexical_ratio_digits_path_url',
 'lexical_ratio_letters_path_url',
 'lexical_count_dots_url',
 'lexical_count_percent_url',
 'lexical_count_hash_url',
 'lexical_count_ats_url',
 'lexical_count_embed_url',
 'lexical_use_https',
 'lexical_no_of_directories',
 'lexical_contains_ip_address',
 'lexical_character_continuity_rate_url',
 'lexical_shannon_entropy_url']

In [8]:
X_test

,lexical_len_url,lexical_len_netloc,lexical_count_digits_netloc,lexical_count_letters_netloc,lexical_ratio_digits_netloc_url,lexical_ratio_letters_netloc_url,lexical_len_path,lexical_count_digits_path,lexical_count_letters_path,lexical_ratio_digits_path_url,...,lexical_count_dots_url,lexical_count_percent_url,lexical_count_hash_url,lexical_count_ats_url,lexical_count_embed_url,lexical_use_https,lexical_no_of_directories,lexical_contains_ip_address,lexical_character_continuity_rate_url,lexical_shannon_entropy_url
4997,11,0,0,0,0.000000,0.000000,11,3,7,0.272727,...,1,0,0,0,0,0,0,0,0.090909,2.845351
8864,12,0,0,0,0.000000,0.000000,12,1,10,0.083333,...,1,0,0,0,0,0,0,0,0.083333,2.751629
2390,34,25,6,16,0.176471,0.470588,1,0,0,0.000000,...,2,0,0,0,1,1,1,0,0.117647,4.348657
6748,41,21,0,19,0.000000,0.463415,13,0,12,0.000000,...,2,0,0,0,1,0,1,0,0.048780,4.122694
3643,13,0,0,0,0.000000,0.000000,13,0,11,0.000000,...,2,0,0,0,0,0,0,0,0.076923,3.238901
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4811,73,43,21,19,0.287671,0.260274,23,0,21,0.000000,...,3,0,0,0,1,0,1,0,0.068493,4.746561
9259,14,0,0,0,0.000000,0.000000,14,0,13,0.000000,...,1,0,0,0,0,0,0,0,0.000000,3.521641
3825,47,27,0,25,0.000000,0.531915,12,0,10,0.000000,...,3,0,0,0,1,1,1,0,0.042553,4.189930
7739,24,0,0,0,0.000000,0.000000,24,0,22,0.000000,...,2,0,0,0,0,0,0,0,0.000000,3.522055


## ScikitLearn Model

In [6]:
# hyperparams for one-hot encoding layer
onehot_hyperparams = {
    'categories': 'auto', 
    'drop': None, 
    'dtype': np.float64, 
    'handle_unknown': 'error', 
    'min_frequency': None, 
    'max_categories': None, 
    'feature_name_combiner': 'concat'
}

# hyperparams for logistic regression models in sklearn
lr_hyperparams = {
    'penalty': 'l2',
    'dual': False,
    'tol': 1e-4,
    'C': 1,
    'fit_intercept': True,
    'intercept_scaling': 1,
    'class_weight': None,
    'random_state': None,
    'solver': 'lbfgs',
    'max_iter': 1000,
    'multi_class': 'auto',
    'verbose': 0,
    'warm_start': False,
    'n_jobs': None,
    'l1_ratio': None
}

In [7]:
# one hot encoder not needed atm
# pipeline accepts X and standardizes (mean/stdev) and passes it to lr model
pipe = Pipeline(
    [
        ('stdscaler', StandardScaler()), 
        #('onehotencoder', OneHotEncoder(**onehot_hyperparams))
        ('lrmodel', LogisticRegression(**lr_hyperparams)),
    ]
)

pipe.fit(X_train, y_train).score(X_test, y_test)

1.0

## Pytorch Model

In [12]:
input_size = X_train.shape[1]
model = LogisticRegressionPytorch(input_size)

_hyperparams = {
    'lr': 0.01, 
    'momentum': 0,
    'dampening': 0,
    'weight_decay': 0,
    'nesterov': False
}

criterion = nn.BCELoss() # cost fxn
optimizer = optim.SGD(model.parameters(), **_hyperparams) # stochastic gradient descent optimizer

num_epochs = 5000

In [13]:
X_train_tensor = torch.from_numpy(X_train.to_numpy().astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.to_numpy().astype(np.float32))

X_test_tensor = torch.from_numpy(X_test.to_numpy().astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.to_numpy().astype(np.float32))

In [14]:
for epoch in range(num_epochs):
    model.train() # training mode activated
    optimizer.zero_grad() # reset gradients

    outputs = model(X_train_tensor) # forward pass to model
    loss = criterion(outputs, y_train_tensor.view(-1, 1)) # compute loss 
    loss.backward() # backprop
    optimizer.step() # optimization step

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [100/5000], Loss: 0.1025
Epoch [200/5000], Loss: 0.0677
Epoch [300/5000], Loss: 0.0502
Epoch [400/5000], Loss: 0.0395
Epoch [500/5000], Loss: 0.0325
Epoch [600/5000], Loss: 0.0276
Epoch [700/5000], Loss: 0.0240
Epoch [800/5000], Loss: 0.0211
Epoch [900/5000], Loss: 0.0185
Epoch [1000/5000], Loss: 0.0164
Epoch [1100/5000], Loss: 0.0145
Epoch [1200/5000], Loss: 0.0130
Epoch [1300/5000], Loss: 0.0117
Epoch [1400/5000], Loss: 0.0106
Epoch [1500/5000], Loss: 0.0097
Epoch [1600/5000], Loss: 0.0089
Epoch [1700/5000], Loss: 0.0083
Epoch [1800/5000], Loss: 0.0077
Epoch [1900/5000], Loss: 0.0072
Epoch [2000/5000], Loss: 0.0067
Epoch [2100/5000], Loss: 0.0063
Epoch [2200/5000], Loss: 0.0060
Epoch [2300/5000], Loss: 0.0057
Epoch [2400/5000], Loss: 0.0054
Epoch [2500/5000], Loss: 0.0051
Epoch [2600/5000], Loss: 0.0049
Epoch [2700/5000], Loss: 0.0047
Epoch [2800/5000], Loss: 0.0045
Epoch [2900/5000], Loss: 0.0043
Epoch [3000/5000], Loss: 0.0041
Epoch [3100/5000], Loss: 0.0040
Epoch [3200/5000]

In [15]:
model.eval() # evaluation mode activated
with torch.no_grad():
    predictions = model(X_test_tensor) # forward pass
    predictions = (predictions > 0.5).float() # thresholding

accuracy = (predictions == y_test_tensor.view(-1, 1)).sum().item() / len(y_test) # compute right/total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 99.99%


In [17]:
# Export the model
torch.onnx.export(model,               # model being run
    X_train_tensor,                         # model input (or a tuple for multiple inputs)
    "model.onnx",   # where to save the model (can be a file or file-like object)
    export_params=True,        # store the trained parameter weights inside the model file
    #opset_version=10,          # the ONNX version to export the model to
    #do_constant_folding=True,  # whether to execute constant folding for optimization
    input_names = ['input'],   # the model's input names
    output_names = ['output'], # the model's output names
    dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                'output' : {0 : 'batch_size'}})